# Ensemble Methods — Exercises

Companion to the note [Ensemble Methods](Ensemble%20Methods.md).

Practise why averaging reduces variance, bootstrap and out-of-bag estimates, random-forest decorrelation, AdaBoost weights, gradient boosting as fitting negative gradients, stacking and feature importance, by hand and from scratch with NumPy and sklearn trees.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×7 · 💻 Code ×5

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from math import comb
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier, GradientBoostingRegressor)
from sklearn.inspection import permutation_importance
from sklearn.datasets import load_breast_cancer
rng = np.random.default_rng(0)
Xbc, ybc = load_breast_cancer(return_X_y=True)
perm = rng.permutation(len(ybc))
tr, te = perm[:400], perm[400:]
Xtr, ytr, Xte, yte = Xbc[tr], ybc[tr], Xbc[te], ybc[te]

## Part A · Concepts

### Exercise 1 · Variance vs bias reducers
*🧠 Concept · ★☆☆*

The note says bagging reduces **variance** and boosting reduces **bias**. Explain why, and say which base learner suits each:
deep trees or stumps? Why does bagging linear regression achieve almost nothing?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Bagging averages models trained *independently* on resamples; boosting adds models *sequentially* to fix what is left.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Bagging averages many noisy but roughly unbiased predictors: the average has the same bias and lower variance (Exercise 5). So it needs **low-bias, high-variance**
base learners: deep, unpruned [[Decision Trees]]. Boosting fits each new model to the current errors, so the sum becomes more flexible step by step,
reducing bias; it works best with **weak, high-bias** learners like stumps or depth-3–8 trees. Linear regression is already low-variance and
its average over bootstrap samples is almost the same linear fit, so there is nothing to gain ([[Bias-Variance Tradeoff]]).

</details>

### Exercise 2 · Why random feature subsets?
*🧠 Concept · ★★☆*

Bagged trees on a dataset with one very strong feature all split on it at the root and look alike. Explain how the random forest's per-split
feature subsampling (`max_features`) helps, using the variance formula of Exercise 5. What is the trade-off when `max_features` is very small?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The variance floor of an average of correlated predictors is $\rho\sigma^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With pairwise correlation $\rho$ the averaged variance tends to $\rho\sigma^2$, not 0, however many trees you add. If all trees share the same top splits, $\rho$ is high.
Restricting each split to a random subset of features forces trees to use different features, lowering $\rho$ (decorrelation). If `max_features` is too small, individual
trees often get only useless features at a split, so their bias and individual variance $\sigma^2$ rise. Defaults: $\sqrt d$ (classification), $d/3$ or $d$ (regression).

</details>

### Exercise 3 · Which feature importance to trust?
*🧠 Concept · ★★☆*

The note prefers **permutation importance** over impurity-based importance. Explain how each is computed, and why impurity importance is biased
towards continuous / high-cardinality features (e.g. a random ID-like column). What is a caveat of permutation importance with correlated features?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Impurity importance is computed on the *training* data; a feature with many distinct values offers many candidate thresholds.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Impurity (MDI)**: sum of weighted impurity decreases of all splits on the feature, averaged over trees, measured on training data. A high-cardinality noise
feature offers many thresholds, so by chance it often achieves some decrease deep in the trees and gets non-zero importance (overfitting shows up as importance).
**Permutation**: shuffle one column of held-out data and measure the drop in score; noise features give ≈ 0 drop. Caveat: if two features are strongly correlated,
permuting one leaves the information in the other, so both look unimportant (and permuting creates unrealistic data points). You will see the bias in Exercise 16.

</details>

### Exercise 4 · Stacking without leakage
*🧠 Concept · ★★☆*

In stacking, a meta-learner is trained on the base models' predictions. Why must these be **out-of-fold** predictions (from cross-validation) rather than
predictions on the data the base models were trained on? Contrast stacking with soft voting.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What does a 1-NN or a deep tree predict on its own training points?

</details>

<details>
<summary><b>✅ Solution</b></summary>

In-sample predictions are over-optimistic: an overfitting base model (deep tree, 1-NN) is perfect on its training data, so the meta-learner would learn to
trust it completely, and fail on new data. Out-of-fold predictions mimic the test-time situation, so the meta-learner learns how reliable each model really is.
Soft voting is the special case with fixed equal weights on predicted probabilities (no meta-learning); stacking *learns* the combination (often a regularised
[[Logistic Regression]]). See [[Cross-Validation and Model Selection]].

</details>

## Part B · By hand

### Exercise 5 · Variance of an average
*✍️ By hand · ★★☆*

$B$ predictors each have variance $\sigma^2$ and pairwise correlation $\rho$. Show that their average has variance $\rho\sigma^2+\frac{1-\rho}{B}\sigma^2$.
Evaluate it for $\sigma^2=1$, $\rho=0.3$, $B=50$, and give the limit for $B\to\infty$.

In [ ]:
var_avg50 = None
var_avg_inf = None

_C = 0.3 * np.ones((50, 50)) + 0.7 * np.eye(50)   # covariance matrix of the 50 predictors
check('variance for B=50', var_avg50, np.ones(50) @ _C @ np.ones(50) / 50**2)
check('limit', var_avg_inf, 0.3)

<details>
<summary><b>💡 Hint</b></summary>

$\operatorname{Var}\big(\frac1B\sum_bf_b\big)=\frac1{B^2}\sum_{b,b'}\operatorname{Cov}(f_b,f_{b'})$; there are $B$ diagonal and $B(B-1)$ off-diagonal terms.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac1{B^2}\big[B\sigma^2+B(B-1)\rho\sigma^2\big]=\frac{\sigma^2}{B}+\rho\sigma^2-\frac{\rho\sigma^2}{B}=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2$.
With the numbers: $0.3+0.7/50=0.314$; as $B\to\infty$ it tends to $0.3$. More trees cannot beat the correlation floor, which is why random forests decorrelate.

```python
var_avg50 = 0.314
var_avg_inf = 0.3
```

</details>

### Exercise 6 · Out-of-bag probability
*✍️ By hand · ★☆☆*

A bootstrap sample draws $N$ points with replacement from $N$. What is the probability that a given point is **not** drawn (it is out-of-bag)?
Evaluate for $N=100$ and the limit $N\to\infty$.

In [ ]:
p_oob_100 = None
p_oob_limit = None

_r = np.random.default_rng(0)
_sim = np.mean([0 not in _r.integers(0, 100, 100) for _ in range(20000)])
print('simulated N=100:', _sim)
check('N=100', p_oob_100, (1 - 1/100)**100, tol=1e-4)
check('limit', p_oob_limit, np.exp(-1), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Each of the $N$ draws misses the point with probability $1-1/N$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(1-1/N)^N$: $0.99^{100}\approx0.366$, and $\to e^{-1}\approx0.368$. So each tree leaves out about 37% of the data, and each point is out-of-bag for about a third
of the trees: predicting it with only those trees gives a free validation estimate (OOB error).

```python
p_oob_100 = 0.99 ** 100
p_oob_limit = np.exp(-1)
```

</details>

### Exercise 7 · Wisdom of (independent) crowds
*✍️ By hand · ★☆☆*

Five classifiers are each correct with probability $0.7$, **independently**. What is the probability that the majority vote is correct? What if they
were perfectly correlated?

In [ ]:
p_majority = None

check('P(majority correct)', p_majority, sum(comb(5, k) * 0.7**k * 0.3**(5-k) for k in range(3, 6)), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

The majority is correct if at least 3 of 5 are correct: a binomial tail.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\binom53 0.7^3 0.3^2+\binom54 0.7^4 0.3+0.7^5=0.3087+0.3602+0.1681=0.8369$. With perfect correlation all five vote the same way, so the vote is correct
with probability $0.7$, no gain. Diversity is what makes ensembles work.

```python
p_majority = 0.83692
```

</details>

### Exercise 8 · One round of AdaBoost
*✍️ By hand · ★★☆*

Five training points have weights $0.2$ each. A stump misclassifies exactly one of them. Compute the weighted error $\varepsilon$, the model weight
$\alpha=\tfrac12\ln\frac{1-\varepsilon}{\varepsilon}$, and the normalised new weights after $w_i\leftarrow w_ie^{-\alpha y_ih(x_i)}$.

In [ ]:
eps_ab = None
alpha_ab = None
w_wrong = None   # new weight of the misclassified point
w_right = None   # new weight of each correctly classified point

_w = np.full(5, 0.2); _m = np.array([1, 0, 0, 0, 0], bool)
_e = _w[_m].sum(); _a = 0.5*np.log((1-_e)/_e)
_wn = _w * np.exp(np.where(_m, _a, -_a)); _wn /= _wn.sum()
check('epsilon', eps_ab, _e); check('alpha', alpha_ab, _a, tol=1e-4)
check('weight of misclassified point', w_wrong, _wn[0]); check('weight of a correct point', w_right, _wn[1])

<details>
<summary><b>💡 Hint</b></summary>

$\varepsilon=0.2$ so $\alpha=\tfrac12\ln4=\ln2$; $e^{\alpha}=2$, $e^{-\alpha}=\tfrac12$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\varepsilon=0.2$, $\alpha=\tfrac12\ln4\approx0.693$. Unnormalised: wrong $0.2\cdot2=0.4$, right $0.2\cdot\tfrac12=0.1$ (four of them, total $0.4$).
Normalised: wrong point $0.5$, each correct point $0.125$. After reweighting, the old stump has weighted error exactly $1/2$: the next learner must find something new.

```python
eps_ab = 0.2
alpha_ab = np.log(2)
w_wrong = 0.5
w_right = 0.125
```

</details>

### Exercise 9 · Gradient boosting by hand
*✍️ By hand · ★☆☆*

Squared loss, targets $y=(3,5,10)$. Initialise $F_0$ as the mean. Compute the pseudo-residuals $r_i=-\partial L/\partial F$ (with $L=\tfrac12(y-F)^2$).
A stump puts points 1 and 2 in one leaf and point 3 in another, predicting the leaf mean of the residuals. With learning rate $\nu=0.1$, compute $F_1$.

In [ ]:
F0_gb = None
res_gb = None  # 3 numbers
F1_gb = None   # 3 numbers

_y = np.array([3, 5, 10.]); _F0 = _y.mean(); _r = _y - _F0
_h = np.array([_r[:2].mean(), _r[:2].mean(), _r[2]])
check('F0', F0_gb, _F0); check('residuals', res_gb, _r); check('F1', F1_gb, _F0 + 0.1*_h)

<details>
<summary><b>💡 Hint</b></summary>

$F_0=6$; the negative gradient of $\tfrac12(y-F)^2$ is $y-F$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$F_0=6$, residuals $(-3,-1,4)$. Leaf values: $-2$ (mean of $-3,-1$) and $4$. $F_1=6+0.1\cdot(-2,-2,4)=(5.8,\ 5.8,\ 6.4)$.
The small learning rate (shrinkage) means many trees are needed, but generalisation is better.

```python
F0_gb = 6.0
res_gb = [-3, -1, 4]
F1_gb = [5.8, 5.8, 6.4]
```

</details>

### Exercise 10 · Pseudo-residuals for log-loss
*✍️ By hand · ★★☆*

For binary classification, gradient boosting models the log-odds $F(x)$ and uses $L=-[y\log p+(1-y)\log(1-p)]$ with $p=\sigma(F)$.
Show that the pseudo-residual is $r=-\partial L/\partial F=y-p$. Compute it for $y=1$, $F=0$ and for $y=0$, $F=2$.

In [ ]:
r_case1 = None
r_case2 = None

check('y=1, F=0', r_case1, 1 - 1/(1+np.exp(-0.)), tol=1e-4)
check('y=0, F=2', r_case2, 0 - 1/(1+np.exp(-2.)), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

This is the same computation as the logistic-regression gradient: $\partial L/\partial F=\sigma(F)-y$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\partial L/\partial F=\sigma(F)-y$ (see [[Logistic Regression]]), so $r=y-p$: the residual on the probability scale. Case 1: $1-0.5=0.5$. Case 2: $0-\sigma(2)\approx-0.881$.
Every loss gives its own 'residual': this is why gradient boosting works with any differentiable loss.

```python
r_case1 = 0.5
r_case2 = -1 / (1 + np.exp(-2))
```

</details>

### Exercise 11 · Where AdaBoost's $\alpha$ comes from
*✍️ By hand · ★★★*

AdaBoost greedily minimises the exponential loss $\sum_iw_ie^{-y_i\alpha h(x_i)}$ over $\alpha$ for a fixed weak learner $h\in\{\pm1\}$ with weighted error $\varepsilon$ (weights sum to 1).
Derive $\alpha^*=\tfrac12\ln\frac{1-\varepsilon}{\varepsilon}$. What happens if $\varepsilon>1/2$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Split the sum into correctly classified points ($y_ih=1$, total weight $1-\varepsilon$) and mistakes (total weight $\varepsilon$).

</details>

<details>
<summary><b>✅ Solution</b></summary>

The loss is $(1-\varepsilon)e^{-\alpha}+\varepsilon e^{\alpha}$. Derivative: $-(1-\varepsilon)e^{-\alpha}+\varepsilon e^{\alpha}=0\Rightarrow e^{2\alpha}=\frac{1-\varepsilon}{\varepsilon}\Rightarrow\alpha^*=\tfrac12\ln\frac{1-\varepsilon}{\varepsilon}$.
At the optimum the loss is $2\sqrt{\varepsilon(1-\varepsilon)}<1$ whenever $\varepsilon\ne\tfrac12$, so the training loss shrinks geometrically.
If $\varepsilon>\tfrac12$, $\alpha<0$: the learner is flipped (worse than chance is still informative). $\varepsilon=\tfrac12$ gives $\alpha=0$: no information.

</details>

## Part C · Code from scratch

### Exercise 12 · Bagging with an OOB estimate
*💻 Code · ★★☆*

Implement `bagging(X, y, B, seed)` with fully grown `DecisionTreeClassifier`s on bootstrap samples. Return the fitted trees and the **OOB accuracy**:
for each training point, majority-vote only the trees that did not see it. Compare the OOB accuracy, the bagged test accuracy and a single tree's test accuracy.

In [ ]:
def bagging(X, y, B=100, seed=0):
    # TODO: return (trees, oob_accuracy)
    return None, None

def bag_predict(trees, X):
    # TODO: majority vote
    return None

trees, oob_acc = bagging(Xtr, ytr)
acc_bag = None if trees is None else np.mean(bag_predict(trees, Xte) == yte)

_single = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr).score(Xte, yte)
print('single tree test acc', _single)
check('bagging beats a single tree', None if acc_bag is None else acc_bag > _single, True)
check('OOB estimate within 4 points of test accuracy', None if oob_acc is None else abs(oob_acc - acc_bag) < 0.04, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Keep a boolean `in_bag[b, i]`. Accumulate votes `votes[i] += pred` and `counts[i] += 1` only where `~in_bag[b]`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Use `DecisionTreeClassifier(random_state=b)` so that results are reproducible.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Bagging improves on the single tree by about two points (0.935 → 0.953), and the OOB accuracy is close to the test accuracy without spending any data on validation.
This is the 'OOB error for free' of the note (`RandomForestClassifier(oob_score=True)`).

```python
def bagging(X, y, B=100, seed=0):
    r = np.random.default_rng(seed)
    n = len(y)
    trees, votes, counts = [], np.zeros(n), np.zeros(n)
    for b in range(B):
        idx = r.integers(0, n, n)
        t = DecisionTreeClassifier(random_state=b).fit(X[idx], y[idx])
        trees.append(t)
        oob = np.ones(n, bool); oob[idx] = False
        votes[oob] += t.predict(X[oob]); counts[oob] += 1
    seen = counts > 0
    oob_acc = np.mean(((votes[seen] / counts[seen]) > 0.5) == y[seen])
    return trees, oob_acc

def bag_predict(trees, X):
    return (np.mean([t.predict(X) for t in trees], axis=0) > 0.5).astype(int)

trees, oob_acc = bagging(Xtr, ytr)
acc_bag = np.mean(bag_predict(trees, Xte) == yte)
print('OOB acc', oob_acc, '| bagged test acc', acc_bag)
```

</details>

### Exercise 13 · AdaBoost with stumps
*💻 Code · ★★☆*

Implement discrete AdaBoost with depth-1 trees (`DecisionTreeClassifier(max_depth=1, random_state=0)` with `sample_weight`), labels $\pm1$, $M=50$ rounds.
Return the stumps and their $\alpha$s, predict with $\operatorname{sign}\sum_m\alpha_mh_m(x)$, and compare with `AdaBoostClassifier`.

In [ ]:
def adaboost(X, y_pm, M=50):
    # TODO: return (stumps, alphas)
    return None, None

def ada_predict(stumps, alphas, X):
    # TODO
    return None

stumps, alphas = adaboost(Xtr, 2 * ytr - 1)
pred_ada = None if stumps is None else ada_predict(stumps, alphas, Xte)

_sk = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=0).fit(Xtr, ytr)
check('agrees with sklearn AdaBoost', None if pred_ada is None else np.mean(pred_ada == 2 * _sk.predict(Xte) - 1) >= 0.98, True)

<details>
<summary><b>💡 Hint</b></summary>

Per round: fit with `sample_weight=w`, `eps = w[h != y].sum() / w.sum()`, `alpha = 0.5*log((1-eps)/eps)`, `w *= exp(-alpha*y*h)`, normalise.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Test accuracy is about 0.96–0.97. sklearn uses the SAMME variant: for two classes its $\alpha$ is exactly twice ours and its weight update differs only by a constant
factor that normalisation removes, so the predictions coincide.

```python
def adaboost(X, y_pm, M=50):
    w = np.full(len(y_pm), 1 / len(y_pm))
    stumps, alphas = [], []
    for m in range(M):
        s = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X, y_pm, sample_weight=w)
        h = s.predict(X)
        eps = np.clip(w[h != y_pm].sum() / w.sum(), 1e-12, 1 - 1e-12)
        a = 0.5 * np.log((1 - eps) / eps)
        w = w * np.exp(-a * y_pm * h); w /= w.sum()
        stumps.append(s); alphas.append(a)
    return stumps, np.array(alphas)

def ada_predict(stumps, alphas, X):
    return np.sign(sum(a * s.predict(X) for s, a in zip(stumps, alphas)))

stumps, alphas = adaboost(Xtr, 2 * ytr - 1)
pred_ada = ada_predict(stumps, alphas, Xte)
print('test accuracy', np.mean(pred_ada == 2 * yte - 1))
```

</details>

### Exercise 14 · Gradient boosting for regression
*💻 Code · ★★☆*

Implement gradient boosting with squared loss: $F_0=\bar y$, then for $m=1..M$ fit `DecisionTreeRegressor(max_depth=3, random_state=0)` to the residuals $y-F_{m-1}$
and update $F_m=F_{m-1}+\nu h_m$. Use $M=100$, $\nu=0.1$ on the noisy sine data below and compare with `GradientBoostingRegressor`.

In [ ]:
xg = np.sort(rng.uniform(0, 6, 200))[:, None]
yg = np.sin(xg[:, 0]) + 0.3 * rng.normal(size=200)

def gb_fit(X, y, M=100, lr=0.1):
    # TODO: return (F0, trees)
    return None

def gb_predict(model, X, lr=0.1):
    # TODO
    return None

gb_model = gb_fit(xg, yg)
pred_gb = None if gb_model is None else gb_predict(gb_model, xg)

check('matches sklearn GradientBoostingRegressor', pred_gb, GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=0).fit(xg, yg).predict(xg), tol=1e-8)

<details>
<summary><b>💡 Hint</b></summary>

Keep a running prediction `F`; each tree is fit to `y - F`, then `F += lr * tree.predict(X)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For squared loss, 'fit the negative gradient' is literally 'fit the residuals', and the optimal leaf values are the residual means, so sklearn's algorithm
reduces to these few lines and the predictions match exactly. For other losses the leaf values come from a line search / Newton step.

```python
xg = np.sort(rng.uniform(0, 6, 200))[:, None]
yg = np.sin(xg[:, 0]) + 0.3 * rng.normal(size=200)

def gb_fit(X, y, M=100, lr=0.1):
    F0 = y.mean()
    F = np.full(len(y), F0)
    trees = []
    for m in range(M):
        t = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X, y - F)
        F += lr * t.predict(X)
        trees.append(t)
    return F0, trees

def gb_predict(model, X, lr=0.1):
    F0, trees = model
    return F0 + lr * sum(t.predict(X) for t in trees)

gb_model = gb_fit(xg, yg)
pred_gb = gb_predict(gb_model, xg)
```

</details>

### Exercise 15 · Decorrelating trees
*💻 Code · ★★☆*

Fit `RandomForestClassifier(n_estimators=100, random_state=0)` twice on the training data: with `max_features=None` (plain bagging) and with `'sqrt'`.
For each forest compute the mean pairwise correlation between the individual trees' predicted probabilities (`est.predict_proba(Xte)[:, 1]`) on the test set.
Store `rho_bag` and `rho_rf` and the two test accuracies.

In [ ]:
rho_bag = None
rho_rf = None

check('feature subsampling lowers correlation', None if rho_bag is None else rho_rf < rho_bag, True)

<details>
<summary><b>💡 Hint</b></summary>

Stack the 100 probability vectors into a (100, n_test) array `P`; `C = np.corrcoef(P)`; average the off-diagonal entries.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The `sqrt` forest has lower inter-tree correlation (≈ 0.83 → ≈ 0.81 here; the effect is modest because many breast-cancer features are themselves highly correlated), which lowers the variance floor $\rho\sigma^2$ of Exercise 5; on this data the test accuracies are similar
or slightly better for the random forest.

```python
def mean_tree_corr(forest, X):
    P = np.array([est.predict_proba(X)[:, 1] for est in forest.estimators_])
    C = np.corrcoef(P)
    return C[~np.eye(len(C), dtype=bool)].mean()

bag = RandomForestClassifier(n_estimators=100, max_features=None, random_state=0).fit(Xtr, ytr)
rf = RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=0).fit(Xtr, ytr)
rho_bag, rho_rf = mean_tree_corr(bag, Xte), mean_tree_corr(rf, Xte)
print(f'bagging: rho={rho_bag:.3f} acc={bag.score(Xte, yte):.3f} | RF: rho={rho_rf:.3f} acc={rf.score(Xte, yte):.3f}')
```

</details>

### Exercise 16 · Permutation importance vs impurity importance
*💻 Code · ★★★*

In the synthetic data below only column 0 (`signal`, binary) carries information (labels are flipped with probability 0.25); column 1 is binary noise
and column 2 is **continuous** noise. Fit `RandomForestClassifier(n_estimators=200, random_state=0)` on the first 500 rows. Implement
`perm_importance(model, X, y, j, n_repeats, seed)`: the mean drop in accuracy on the last 500 rows when column $j$ is shuffled.
Store the impurity importances (`feature_importances_`) in `mdi` and your permutation importances of the three columns in `pi`.

In [ ]:
r_imp = np.random.default_rng(0)
signal = r_imp.integers(0, 2, 1000)
y_imp = np.where(r_imp.uniform(size=1000) < 0.25, 1 - signal, signal)
X_imp = np.c_[signal, r_imp.integers(0, 2, 1000), r_imp.normal(size=1000)]

def perm_importance(model, X, y, j, n_repeats=10, seed=0):
    # TODO
    return None

mdi = None   # 3 impurity importances
pi = None    # 3 permutation importances

check('MDI ranks the continuous noise column first', None if mdi is None else int(np.argmax(mdi)) == 2, True)
check('permutation importance ranks the signal first', None if pi is None else int(np.argmax(pi)) == 0, True)

<details>
<summary><b>💡 Hint</b></summary>

Shuffle with `Xp = X.copy(); Xp[:, j] = r.permutation(Xp[:, j])`; importance = baseline score minus permuted score, on the held-out rows.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Impurity importance gives the continuous noise column about 0.7 and the real signal only about 0.3: the fully grown trees use the hundreds of thresholds of the noise
feature to fit the flipped labels, and MDI rewards every such training-set split. Permutation importance on held-out data ranks the signal first (≈ 0.15) and the noise
columns near zero (≈ 0.02, small but non-zero because the overfitted trees do rely on them). This is the bias described in Exercise 3.

```python
r_imp = np.random.default_rng(0)
signal = r_imp.integers(0, 2, 1000)
y_imp = np.where(r_imp.uniform(size=1000) < 0.25, 1 - signal, signal)
X_imp = np.c_[signal, r_imp.integers(0, 2, 1000), r_imp.normal(size=1000)]

def perm_importance(model, X, y, j, n_repeats=10, seed=0):
    r = np.random.default_rng(seed)
    base = model.score(X, y)
    drops = []
    for _ in range(n_repeats):
        Xp = X.copy(); Xp[:, j] = r.permutation(Xp[:, j])
        drops.append(base - model.score(Xp, y))
    return float(np.mean(drops))

rf_imp = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_imp[:500], y_imp[:500])
mdi = rf_imp.feature_importances_
pi = np.array([perm_importance(rf_imp, X_imp[500:], y_imp[500:], j) for j in range(3)])
print('MDI', mdi.round(3), '| permutation', pi.round(3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Ensemble Methods](Ensemble%20Methods.md).*